# Post-Flood Rescue and Ration Distribution Management System\n\nMulti-output Random Forest model for camp-level rescue and ration priority prediction.

## 1. Install and import libraries

In [ ]:
!pip install pandas scikit-learn joblib numpy\n\nimport joblib\nimport numpy as np\nimport pandas as pd\nfrom google.colab import files\nfrom sklearn.ensemble import RandomForestClassifier\nfrom sklearn.metrics import accuracy_score, classification_report, confusion_matrix\nfrom sklearn.model_selection import train_test_split\nfrom sklearn.multioutput import MultiOutputClassifier\nfrom sklearn.preprocessing import LabelEncoder

## 2. Upload dataset

In [ ]:
uploaded = files.upload()\ndataset_path = list(uploaded.keys())[0]\nprint('Uploaded dataset:', dataset_path)

## 3. Load dataset

In [ ]:
df = pd.read_csv(dataset_path)

## 4. Dataset overview

In [ ]:
print('Dataset shape:', df.shape)\nprint('Column names:')\nprint(df.columns.tolist())\n\nprint('First 5 rows:')\ndisplay(df.head())\n\nprint('Missing values:')\nprint(df.isnull().sum())\n\nif 'human_error_flag' in df.columns:\n    error_count = (df['human_error_flag'].astype(str).str.lower() == 'yes').sum()\n    print('Human-error rows:', error_count)

## 5. Data cleaning

In [ ]:
feature_columns = [\n    'population',\n    'children_count',\n    'elderly_count',\n    'food_available',\n    'water_available',\n    'medicine_available',\n    'sanitary_available',\n    'distance_from_distribution_center',\n    'camp_capacity',\n    'road_access_status',\n]\n\ntarget_columns = [\n    'camp_priority',\n    'food_priority',\n    'water_priority',\n    'medicine_priority',\n    'sanitary_priority',\n]\n\nvalid_road_status = ['Good', 'Limited', 'Blocked']\n\nclean_df = df.copy()\n\nif 'human_error_flag' in clean_df.columns:\n    clean_df = clean_df[clean_df['human_error_flag'].astype(str).str.lower() != 'yes'].copy()\n\nbefore_rows = len(clean_df)\n\nclean_df = clean_df.dropna(subset=feature_columns + target_columns).copy()\nclean_df = clean_df[clean_df['population'] > 0]\nclean_df = clean_df[clean_df['children_count'] >= 0]\nclean_df = clean_df[clean_df['elderly_count'] >= 0]\nclean_df = clean_df[(clean_df['children_count'] + clean_df['elderly_count']) <= clean_df['population']]\nclean_df = clean_df[clean_df['camp_capacity'] > 0]\nclean_df = clean_df[clean_df['food_available'] >= 0]\nclean_df = clean_df[clean_df['water_available'] >= 0]\nclean_df = clean_df[clean_df['medicine_available'] >= 0]\nclean_df = clean_df[clean_df['sanitary_available'] >= 0]\nclean_df = clean_df[clean_df['distance_from_distribution_center'] >= 0]\nclean_df = clean_df[clean_df['road_access_status'].isin(valid_road_status)]\n\nprint('Rows before cleaning:', before_rows)\nprint('Rows after cleaning:', len(clean_df))

## 6. Feature and target selection

In [ ]:
X = clean_df[feature_columns].copy()\ny = clean_df[target_columns].copy()\n\nprint('Input columns:', feature_columns)\nprint('Target columns:', target_columns)

## 7. Encoding

In [ ]:
feature_encoders = {}\ntarget_encoders = {}\n\nroad_encoder = LabelEncoder()\nX['road_access_status'] = road_encoder.fit_transform(X['road_access_status'])\nfeature_encoders['road_access_status'] = road_encoder\n\nfor column in target_columns:\n    encoder = LabelEncoder()\n    y[column] = encoder.fit_transform(y[column])\n    target_encoders[column] = encoder\n\nlabel_data = {\n    'feature_encoders': feature_encoders,\n    'target_encoders': target_encoders,\n    'input_columns': feature_columns,\n    'target_columns': target_columns,\n}

## 8. Train-test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(\n    X,\n    y,\n    test_size=0.2,\n    random_state=42,\n)\n\nprint('Training rows:', len(X_train))\nprint('Testing rows:', len(X_test))

## 9. Multi-output Random Forest model training

In [ ]:
base_model = RandomForestClassifier(n_estimators=200, random_state=42)\nmodel = MultiOutputClassifier(base_model)\n\nmodel.fit(X_train, y_train)\nprint('Training completed')

## 10. Model evaluation

In [ ]:
y_pred = model.predict(X_test)\n\nfor index, column in enumerate(target_columns):\n    actual = y_test.iloc[:, index]\n    predicted = y_pred[:, index]\n    encoder = target_encoders[column]\n\n    print('\\n' + '=' * 50)\n    print(column)\n    print('=' * 50)\n    print('Accuracy:', accuracy_score(actual, predicted))\n    print('Classification report:')\n    print(classification_report(actual, predicted, target_names=encoder.classes_))\n    print('Confusion matrix:')\n    print(confusion_matrix(actual, predicted))

## 11. Sample prediction

In [ ]:
sample_input = {\n    'population': 420,\n    'children_count': 110,\n    'elderly_count': 60,\n    'food_available': 90,\n    'water_available': 200,\n    'medicine_available': 15,\n    'sanitary_available': 45,\n    'distance_from_distribution_center': 8,\n    'camp_capacity': 600,\n    'road_access_status': 'Limited',\n}\n\nsample_df = pd.DataFrame([sample_input])\nsample_df['road_access_status'] = road_encoder.transform(sample_df['road_access_status'])\nsample_df = sample_df[feature_columns]\n\nsample_prediction = model.predict(sample_df)[0]\nprediction_result = {}\n\nfor index, column in enumerate(target_columns):\n    encoder = target_encoders[column]\n    prediction_result[column] = encoder.inverse_transform([sample_prediction[index]])[0]\n\nprediction_result

## 12. Save model and encoders

In [ ]:
joblib.dump(model, 'camp_relief_priority_model.pkl')\njoblib.dump(label_data, 'label_encoders.pkl')\n\nprint('Saved camp_relief_priority_model.pkl')\nprint('Saved label_encoders.pkl')

## 13. Download .pkl files

In [ ]:
files.download('camp_relief_priority_model.pkl')\nfiles.download('label_encoders.pkl')